# Data Cleaning & Structural Validation
**Dataset:** `customer_churn_raw_v2.csv` (customer churn)

**Objective:** remove duplicates, handle missing values, fix outliers and validate data types. Every decision is documented.

In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv('customer_churn_raw_v2.csv')
df = raw.copy()
log = []   # cleaning decision log
print(raw.shape)
raw.head()

(605, 13)


    CustomerID  Age   Region  TenureMonths SubscriptionType  MonthlyChargesINR TotalChargesINR    ContractType  \
0  CUST-100001   20      NaN             9              Pro           13097.76         unknown  Month-to-Month   
1  CUST-100002   55  Central            52              Pro            1101.18        57334.22        One Year   
2  CUST-100003   40  Central            67         Standard             692.66        44554.47  Month-to-Month   
3  CUST-100004   53    South            48              Pro            1156.49        53956.12        One Year   
4  CUST-100005   21     West            24         Standard             754.16        18191.59  Month-to-Month   

   SupportTickets  LatePayments PaymentMethod AutoPay Churn  
0               1             2          Card      No   Yes  
1               0             0          Card     Yes    No  
2               6             2           UPI      No   Yes  
3               2             2          Card     Yes    No  
4    

## 1. Inspect the raw data

In [2]:
raw.info()
print('\nMissing values:\n', raw.isnull().sum()[lambda s: s > 0])
print('\nFully duplicated rows:', raw.duplicated().sum())
print('Duplicate CustomerIDs:', raw.CustomerID.duplicated().sum())
print('\nTotalChargesINR non-numeric values:')
print(raw.TotalChargesINR[pd.to_numeric(raw.TotalChargesINR, errors='coerce').isna()].value_counts())
raw.describe().T

<class 'pandas.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         605 non-null    str    
 1   Age                605 non-null    int64  
 2   Region             592 non-null    str    
 3   TenureMonths       605 non-null    int64  
 4   SubscriptionType   605 non-null    str    
 5   MonthlyChargesINR  605 non-null    float64
 6   TotalChargesINR    605 non-null    str    
 7   ContractType       605 non-null    str    
 8   SupportTickets     605 non-null    int64  
 9   LatePayments       605 non-null    int64  
 10  PaymentMethod      605 non-null    str    
 11  AutoPay            605 non-null    str    
 12  Churn              605 non-null    str    
dtypes: float64(1), int64(4), str(8)
memory usage: 61.6 KB

Missing values:
 Region    13
dtype: int64

Fully duplicated rows: 5
Duplicate CustomerIDs: 5

TotalChargesINR non-numeric val

                   count        mean          std     min     25%     50%      75%       max
Age                605.0   42.578512    14.765469   18.00   30.00   42.00    55.00     68.00
TenureMonths       605.0   35.552066    20.948360    0.00   18.00   34.00    54.00     72.00
MonthlyChargesINR  605.0  868.016347  1104.807037  364.34  451.57  726.22  1103.47  13858.56
SupportTickets     605.0    1.757025     1.426617    0.00    1.00    2.00     3.00      6.00
LatePayments       605.0    1.277686     1.066768    0.00    0.00    1.00     2.00      4.00

**Issues found**
- 5 exact duplicate rows (605 rows but only 600 unique CustomerIDs)
- `Region` has 13 missing values
- `TotalChargesINR` is stored as text because 11 rows contain the string `unknown`
- `MonthlyChargesINR` has 6 extreme outliers (about 12.5x too high for their plan) plus 1 more in a row with unknown total
- `TenureMonths = 0` for some customers (valid: brand-new customers)

## 2. Remove duplicates
The duplicate rows are identical in every column, so keeping the first occurrence loses no information.

In [3]:
before = len(df)
df = df.drop_duplicates(keep='first').reset_index(drop=True)
log.append(('Duplicates', f'Dropped {before-len(df)} exact duplicate rows', 'Identical in all columns; keeping first loses nothing'))
assert df.CustomerID.is_unique
print(before, '->', len(df))

605 -> 600


## 3. Standardize headers and text values

In [4]:
import re
def to_snake(c):
    c = re.sub(r'(?<=[a-z])(?=[A-Z])', '_', c)
    return c.lower()
df.columns = [to_snake(c) for c in df.columns]
log.append(('Headers', 'Converted column names to snake_case', 'Consistent, easy to use in code'))

text_cols = df.select_dtypes(include=['object', 'string']).columns
for c in text_cols:
    df[c] = df[c].str.strip()
for c in ['subscription_type', 'contract_type', 'payment_method', 'region']:
    df[c] = df[c].str.title()
df['payment_method'] = df['payment_method'].replace({'Upi': 'UPI'})
log.append(('Categoricals', 'Trimmed whitespace and applied consistent casing', 'Prevents the same category appearing under different spellings'))
df.columns.tolist()

['customer_id', 'age', 'region', 'tenure_months', 'subscription_type', 'monthly_charges_inr', 'total_charges_inr', 'contract_type', 'support_tickets', 'late_payments', 'payment_method', 'auto_pay', 'churn']

## 4. Fix data types and the `unknown` placeholder
`total_charges_inr` must be numeric. `unknown` is a placeholder for a missing value, so it is converted to `NaN` first.

In [5]:
df['total_charges_inr'] = pd.to_numeric(df['total_charges_inr'].replace('unknown', np.nan), errors='coerce')
log.append(('Data types', "Converted total_charges_inr to float; 'unknown' -> NaN", 'Text placeholder blocked numeric analysis'))

for c in ['auto_pay', 'churn']:
    df[c] = df[c].map({'Yes': 1, 'No': 0}).astype('int8')
log.append(('Data types', 'Encoded auto_pay and churn as 1/0', 'Binary flags are easier for analysis and modelling'))
df.dtypes

customer_id                str
age                      int64
region                     str
tenure_months            int64
subscription_type          str
monthly_charges_inr    float64
total_charges_inr      float64
contract_type              str
support_tickets          int64
late_payments            int64
payment_method             str
auto_pay                  int8
churn                     int8
dtype: object

## 5. Handle outliers in `monthly_charges_inr`
For each subscription type the 95th percentile is within about 6% of the median, so prices are tightly clustered. Seven values are roughly 10-30x higher, and in six of them `total / (monthly x tenure)` is about 0.08 instead of about 1.0, so the monthly value is clearly inflated (a data-entry or scaling error). Those values are replaced with the median monthly charge of the same plan (median is robust to the outliers themselves).

In [6]:
grp = df.groupby('subscription_type')['monthly_charges_inr']
med = grp.transform('median')
q3 = grp.transform(lambda s: s.quantile(0.75))
iqr = grp.transform(lambda s: s.quantile(0.75) - s.quantile(0.25))
out = df['monthly_charges_inr'] > (q3 + 3 * iqr)
print('Outliers flagged:', out.sum())
display(df.loc[out, ['customer_id', 'subscription_type', 'tenure_months', 'monthly_charges_inr', 'total_charges_inr']])

df.loc[out, 'monthly_charges_inr'] = med[out].round(2)
log.append(('Outliers', f'Replaced {out.sum()} inflated monthly_charges_inr values with the median of their subscription type', 'About 12.5x too high vs plan price and vs total charges / tenure; median is robust'))
df.groupby('subscription_type')['monthly_charges_inr'].describe()

Outliers flagged: 7


     customer_id subscription_type  tenure_months  monthly_charges_inr  total_charges_inr
0    CUST-100001               Pro              9             13097.76                NaN
89   CUST-100090          Standard             42              9374.16           32698.72
178  CUST-100179               Pro              3             13239.48            3189.06
267  CUST-100268             Basic             65              5164.56           27928.48
356  CUST-100357          Standard             32              8863.80           22928.73
445  CUST-100446          Standard             57              8011.20           36568.18
534  CUST-100535               Pro             36             13858.56           39825.11

                   count         mean        std      min        25%      50%        75%      max
subscription_type                                                                                
Basic              211.0   424.087536  35.368607   364.34   393.0650   420.54   455.2650   483.55
Pro                210.0  1127.425429  35.113106  1064.16  1100.3625  1127.86  1158.7575  1183.78
Standard           179.0   723.749944  34.649895   664.11   693.0900   725.99   753.3400   782.77

## 6. Handle missing values
**`total_charges_inr`** (11 missing): total charges should equal monthly charge x tenure. For existing customers with valid totals the ratio is about 0.98 (median), so the estimate is reliable. Customers with 0 months tenure get 0. This is better than the mean or dropping rows (dropping would lose valid customers).

**`region`** (missing): it is a nominal category, so no value can be inferred honestly. It is filled with `Unknown` instead of guessing, which also avoids biasing regional analysis.

In [7]:
miss = df['total_charges_inr'].isna()
df.loc[miss, 'total_charges_inr'] = (df.loc[miss, 'monthly_charges_inr'] * df.loc[miss, 'tenure_months']).round(2)
log.append(('Missing values', f'Imputed {miss.sum()} total_charges_inr as monthly_charges x tenure', 'Total charges is cumulative billing; ratio to monthly x tenure is about 0.98 in valid rows'))

r_missing = df['region'].isna().sum()
df['region'] = df['region'].fillna('Unknown')
log.append(('Missing values', f'Filled {r_missing} missing region values with "Unknown"', 'Nominal category cannot be inferred; avoids fabricating data'))
print(df.isnull().sum().sum(), 'missing values left')

0 missing values left


## 7. Validation checks

In [8]:
checks = {
 'No missing values': df.isnull().sum().sum() == 0,
 'No duplicate rows': df.duplicated().sum() == 0,
 'customer_id unique': df.customer_id.is_unique,
 'age in 18-100': df.age.between(18, 100).all(),
 'tenure 0-72': df.tenure_months.between(0, 72).all(),
 'charges non-negative': (df[['monthly_charges_inr', 'total_charges_inr']] >= 0).all().all(),
 'total approx monthly x tenure (+-25%)': (df.total_charges_inr.sub(df.monthly_charges_inr * df.tenure_months).abs() <= 0.25 * (df.monthly_charges_inr * df.tenure_months) + 1).all(),
 'numeric dtypes': pd.api.types.is_float_dtype(df.total_charges_inr),
}
for k, v in checks.items():
    print('PASS' if v else 'FAIL', '-', k)
assert all(checks.values())

PASS - No missing values
PASS - No duplicate rows
PASS - customer_id unique
PASS - age in 18-100
PASS - tenure 0-72
PASS - charges non-negative
PASS - total approx monthly x tenure (+-25%)
PASS - numeric dtypes


## 8. Export

In [9]:
df.to_csv('customer_churn_cleaned.csv', index=False)
log_df = pd.DataFrame(log, columns=['Area', 'Action', 'Reasoning'])
log_df.to_csv('cleaning_log.csv', index=False)
print(f'Raw: {raw.shape}  ->  Cleaned: {df.shape}')
log_df

Raw: (605, 13)  ->  Cleaned: (600, 13)


             Area                                             Action  \
0      Duplicates                     Dropped 5 exact duplicate rows   
1         Headers               Converted column names to snake_case   
2    Categoricals   Trimmed whitespace and applied consistent casing   
3      Data types  Converted total_charges_inr to float; 'unknown...   
4      Data types                  Encoded auto_pay and churn as 1/0   
5        Outliers  Replaced 7 inflated monthly_charges_inr values...   
6  Missing values  Imputed 10 total_charges_inr as monthly_charge...   
7  Missing values     Filled 13 missing region values with "Unknown"   

                                           Reasoning  
0  Identical in all columns; keeping first loses ...  
1                    Consistent, easy to use in code  
2  Prevents the same category appearing under dif...  
3          Text placeholder blocked numeric analysis  
4  Binary flags are easier for analysis and model...  
5  About 12.5x too hi

## Summary
| Issue | Count | Action |
|---|---|---|
| Duplicate rows | 5 | Dropped (kept first) |
| `unknown` in total charges | 10 after dedup | Converted to NaN, imputed monthly x tenure |
| Missing region | see log | Filled with `Unknown` |
| Monthly charge outliers | 7 | Replaced with plan median |
| Text columns / headers | all | Trimmed, consistent case, snake_case |
| Data types | 3 | total charges to float, yes/no flags to 1/0 |

Final dataset: 600 rows x 13 columns, with no missing values or duplicates.